In [1]:
import pandas as pd
import sqlite3
import os

# Load the data
df = pd.read_csv("../data/superstore.csv", encoding="latin-1")
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

# Create a file-based SQL database
conn = sqlite3.connect("../data/superstore.db")
df.to_sql("superstore", conn, index=False, if_exists="replace")

print("Database ready. File saved to data/superstore.db")

Database ready. File saved to data/superstore.db


In [2]:
# Helper function to run SQL queries cleanly
def run_query(query):
    return pd.read_sql_query(query, conn)

In [3]:
#1- Total sales , Profit and Orders overview 
run_query("""
          SELECT
            COUNT(*) AS Total_Orders,
            ROUND(SUM(Sales), 2) AS Total_Sales,
            ROUND(SUM(Profit), 2) AS Total_Profit,
            ROUND(AVG(Sales), 2) AS Avg_Sales_per_Order,
            ROUND(SUM(Profit) / SUM(Sales) * 100, 2) AS Profit_Margin_Percentage
          FROM superstore
        """)

,Total_Orders,Total_Sales,Total_Profit,Avg_Sales_per_Order,Profit_Margin_Percentage
0,9994,2297200.86,286397.02,229.86,12.47


## Business Overview

| Metric | Value |
|---|---|
| Total Orders | 9,994 |
| Total Sales | $2,297,200 |
| Total Profit | $286,397 |
| Avg Order Value | $229.86 |
| Overall Profit Margin | 12.47% |

**Finding:** The business generated $2.3M in revenue over 4 years 
but kept only 12.47 cents per dollar — dragged down by Furniture's 2.49% margin.

In [9]:
# Top 10 profitable products
run_query("""
          SELECT
            "Product Name",
            SUM(Sales) AS Total_Sales,
            SUM(Profit) AS Total_Profit,
            ROUND(SUM(Profit) / SUM(Sales) * 100, 2) AS Profit_Margin_Percentage,
            COUNT(*) AS Total_Orders
          FROM superstore
          GROUP BY "Product Name"
          ORDER BY Total_Profit DESC
          LIMIT 10
        """)

,Product Name,Total_Sales,Total_Profit,Profit_Margin_Percentage,Total_Orders
0,Canon imageCLASS 2200 Advanced Copier,61599.824,25199.9280,40.91,5
1,Fellowes PB500 Electric Punch Plastic Comb Bin...,27453.384,7753.0390,28.24,10
2,Hewlett Packard LaserJet 3310 Copier,18839.686,6983.8836,37.07,8
3,Canon PC1060 Personal Laser Copier,11619.834,4570.9347,39.34,4
4,HP Designjet T520 Inkjet Large Format Printer ...,18374.895,4094.9766,22.29,3
5,Ativa V4110MDD Micro-Cut Shredder,7699.890,3772.9461,49.00,2
6,"3D Systems Cube Printer, 2nd Generation, Magenta",14299.890,3717.9714,26.00,2
7,Plantronics Savi W720 Multi-Device Wireless He...,9367.290,3696.2820,39.46,7
8,Ibico EPK-21 Electric Binding System,15875.916,3345.2823,21.07,3
9,Zebra ZM400 Thermal Label Printer,6965.700,3343.5360,48.00,2


## Key Insight — Top 10 Most Profitable Products

- All top 10 products are Technology items
- Canon imageCLASS 2200 leads with $25k profit at 40.9% margin from only 5 orders
- High-margin products (40-49%) are sold in low volumes but drive significant profit
- The business should protect pricing on these products and avoid discounting them

**Recommendation:** Never apply discounts to top performing Technology products —
they are already the most profitable and discounting would directly destroy margin.

In [11]:
# 2. Top 10 Least Profitable Products
run_query("""
    SELECT 
        "Product Name",
        ROUND(SUM(Sales), 2) AS total_sales,
        ROUND(SUM(Profit), 2) AS total_profit,
        ROUND(SUM(Profit) / SUM(Sales) * 100, 2) AS profit_margin,
        COUNT(*) AS total_orders
    FROM superstore
    GROUP BY "Product Name"
    ORDER BY total_profit ASC 
    LIMIT 10
    """)

,Product Name,total_sales,total_profit,profit_margin,total_orders
0,Cubify CubeX 3D Printer Double Head Print,11099.96,-8879.97,-80.00,3
1,Lexmark MX611dhe Monochrome Laser Printer,16829.90,-4589.97,-27.27,4
2,Cubify CubeX 3D Printer Triple Head Print,7999.98,-3839.99,-48.00,1
3,Chromcraft Bull-Nose Wood Oval Conference Tabl...,9917.64,-2876.12,-29.00,5
4,Bush Advantage Collection Racetrack Conference...,9544.73,-1934.40,-20.27,7
5,GBC DocuBind P400 Electric Binding System,17965.07,-1878.17,-10.45,6
6,Cisco TelePresence System EX90 Videoconferenci...,22638.48,-1811.08,-8.00,1
7,Martin Yale Chadless Opener Electric Letter Op...,16656.20,-1299.18,-7.80,6
8,Balt Solid Wood Round Tables,6518.75,-1201.06,-18.42,4
9,BoxOffice By Design Rectangular and Half-Moon ...,1706.25,-1148.44,-67.31,3


## Key Insight — Bottom 10 Loss-Making Products

- Cubify CubeX 3D Printer loses 80% margin — the worst performing product in the entire catalog
- Furniture products dominate the loss list — conference tables and round tables consistently lose money
- Even some Technology products are loss-making when discounted heavily

**Recommendation:** 
1. Immediately discontinue or reprice Cubify 3D printers
2. Review all conference table pricing — selling them at current prices guarantees a loss
3. Set a strict minimum price floor across all products to prevent below-cost selling

In [12]:
# 3. Sales and Profit by Customer Segment
run_query("""
    SELECT 
        Segment,
        COUNT(*) AS total_orders,
        ROUND(SUM(Sales), 2) AS total_sales,
        ROUND(SUM(Profit), 2) AS total_profit,
        ROUND(SUM(Profit) / SUM(Sales) * 100, 2) AS profit_margin,
        ROUND(AVG(Sales), 2) AS avg_order_value
    FROM superstore
    GROUP BY Segment
    ORDER BY total_profit DESC
    """)

,Segment,total_orders,total_sales,total_profit,profit_margin,avg_order_value
0,Consumer,5191,1161401.34,134119.21,11.55,223.73
1,Corporate,3020,706146.37,91979.13,13.03,233.82
2,Home Office,1783,429653.15,60298.68,14.03,240.97


## Key Insight — Customer Segment Analysis

| Segment | Orders | Total Sales | Profit Margin | Avg Order Value |
|---|---|---|---|---|
| Consumer | 5,191 | $1,161,401 | 11.55% | $223.73 |
| Corporate | 3,020 | $706,146 | 13.03% | $233.82 |
| Home Office | 1,783 | $429,653 | 14.03% | $240.97 |

**Finding:** Home Office customers generate the highest margin and highest 
average order value despite being the smallest segment.

**Recommendation:** Invest more in acquiring Home Office and Corporate customers
— they are more profitable per order than Consumer customers.

In [13]:
# 4. Top 10 most valuable customers
run_query("""
    SELECT 
        "Customer Name",
        Segment,
        Region,
        COUNT(*) AS total_orders,
        ROUND(SUM(Sales), 2) AS total_sales,
        ROUND(SUM(Profit), 2) AS total_profit,
        ROUND(SUM(Profit) / SUM(Sales) * 100, 2) AS profit_margin
    FROM superstore
    GROUP BY "Customer Name"
    ORDER BY total_profit DESC
    LIMIT 10
    """)

,Customer Name,Segment,Region,total_orders,total_sales,total_profit,profit_margin
0,Tamara Chand,Corporate,West,12,19052.22,8981.32,47.14
1,Raymond Buch,Consumer,East,18,15117.34,6976.10,46.15
2,Sanjit Chand,Consumer,West,22,14142.33,5757.41,40.71
3,Hunter Lopez,Consumer,Central,11,12873.30,5622.43,43.68
4,Adrian Barton,Consumer,West,20,14473.57,5444.81,37.62
5,Tom Ashbrook,Home Office,East,10,14595.62,4703.79,32.23
6,Christopher Martinez,Consumer,South,10,8954.02,3899.89,43.55
7,Keith Dawkins,Corporate,East,28,8181.26,3038.63,37.14
8,Andy Reiter,Consumer,East,9,6608.45,2884.62,43.65
9,Daniel Raglin,Home Office,East,13,8350.87,2869.08,34.36


## Key Insight — Top 10 Most Valuable Customers

- Tamara Chand leads with $8,981 profit at 47.14% margin
- Top customers are concentrated in West and East regions
- Consumer segment produces the most high-value individual customers
  despite having the lowest overall segment margin
- Central region is underrepresented in top customers — only 1 out of 10

**Recommendation:** Build a customer retention strategy around top performers.
Losing Tamara Chand or Raymond Buch would directly impact profitability.

In [14]:
# 5. Discount impact on Profit
run_query("""
    SELECT 
        CASE 
            WHEN Discount = 0 THEN '0% - No Discount'
            WHEN Discount <= 0.1 THEN '1% - 10%'
            WHEN Discount <= 0.2 THEN '11% - 20%'
            WHEN Discount <= 0.3 THEN '21% - 30%'
            WHEN Discount <= 0.4 THEN '31% - 40%'
            ELSE '40%+'
        END AS discount_range,
        COUNT(*) AS total_orders,
        ROUND(SUM(Sales), 2) AS total_sales,
        ROUND(SUM(Profit), 2) AS total_profit,
        ROUND(SUM(Profit) / SUM(Sales) * 100, 2) AS profit_margin
    FROM superstore
    GROUP BY discount_range
    ORDER BY total_profit DESC
    """)

,discount_range,total_orders,total_sales,total_profit,profit_margin
0,0% - No Discount,4798,1087908.47,320987.60,29.51
1,11% - 20%,3709,792152.89,91756.30,11.58
2,1% - 10%,94,54369.35,9029.18,16.61
3,21% - 30%,227,103226.65,-10369.28,-10.05
4,31% - 40%,233,130911.24,-25448.19,-19.44
5,40%+,933,128632.25,-99558.59,-77.40


## Key Insight — Discount Impact on Profit

- Orders with zero discount generate 29.51% profit margin
- Profit margin drops consistently as discount increases
- **Break-even point: 20% discount** — anything above guarantees a loss
- 40%+ discount orders lose 77.40% — the company is paying customers to take products

**This is the root cause of every problem found in this analysis:**
- Furniture losses → over-discounted
- Binders losses → over-discounted  
- Central region weakness → over-discounted
- Loss-making products → over-discounted

**Recommendation:** Implement a hard cap of 20% maximum discount 
across all products and regions immediately.